In [1]:
import pandas as pd
recipes_df = pd.read_csv("recipes_df.csv")

In [2]:
recipes_df

,Unnamed: 0,Title,Ingredients,Photo
0,0,Cookbook:1-2-3-4 Cake,"['1 cup (240 g / 8.5 oz ) butter', '1 cup (240...",https://en.wikibooks.org/wiki/Special:FilePath...
1,1,Cookbook:A Nice Cup of Tea,"['32 fl oz (1 L) hot water', '3–5 measures of ...",https://en.wikibooks.org/wiki/Special:FilePath...
2,2,Cookbook:Aadun (Nigerian Corn Flour with Palm ...,"['1 cup corn flour', '¼ cup palm oil', '½ teas...",https://en.wikibooks.org/wiki/Special:FilePath...
3,3,User:Aaron Liu/sandbox,"['5 eggs', '80 grams (⅓ cup ) unsalted margari...",https://en.wikibooks.org/wiki/Special:FilePath...
4,4,Cookbook:Abula (Nigerian Three Stews),"['1 cup dried beans', '2 large pieces of meat ...",https://en.wikibooks.org/wiki/Special:FilePath...
...,...,...,...,...
792,792,Cookbook:Zucchini Pie,"['1⅓ cup flour', '½ teaspoon salt', '½ cup sho...",https://en.wikibooks.org/wiki/Special:FilePath...
793,793,Cookbook:Zupa Ogórkowa (Polish Cucumber Soup),"['4 chicken wings', '1 small leek (or 1 small ...",https://en.wikibooks.org/wiki/Special:FilePath...
794,794,Cookbook:Zuppa Toscana,"['1 lb spicy Italian sausage , crumbled', '½ l...",https://en.wikibooks.org/wiki/Special:FilePath...
795,795,Cookbook:Zwiebelrostbraten with Red Wine Onion...,"['1 onion , finely chopped', '2 garlic cloves,...",https://en.wikibooks.org/wiki/Special:FilePath...


# Ingredients extraction

In [3]:
import requests
import ast
import tqdm

def extract_ingredient(ingredients_string):
    ingredients = []
    ingredients_list = ast.literal_eval(ingredients_string)
    for ingredient in tqdm.tqdm(ingredients_list, desc="Extracting ingredients"):
        URL = "http://localhost:11434/api/generate"
        prompt = f"Extract the main ingredient from the string: {ingredient.strip()}\nReturn only the main ingredient without any additional text. For example, if the input is '2 cups of flour', the output should be 'flour'."
        payload = {
            "model": "llama3.2:3b",
            "prompt": prompt,
            "stream": False
        }
        response = requests.post(URL, json=payload, timeout=180)
        if response.status_code == 200:
            ingredients.append(response.json()["response"].strip())
        else:
            print(f"Error: {response.status_code} - {response.text}")
    return ingredients

In [4]:
import ast
ast.literal_eval(recipes_df["Ingredients"].iloc[0])


['1 cup (240 g / 8.5 oz ) butter',
 '1 cup (240 ml / 8.1 oz) milk',
 '1 teaspoon vanilla extract',
 '2 cups (450 g / 16 oz) white granulated sugar',
 '3 cups (400 g / 14 oz) all-purpose flour',
 '3 teaspoons baking powder',
 '3 pinches of salt',
 '4 eggs']

In [6]:
extract_ingredient(recipes_df["Ingredients"].iloc[0])

Extracting ingredients: 100%|██████████| 8/8 [00:26<00:00,  3.26s/it]


['butter',
 'milk',
 'vanilla',
 'sugar',
 'flour',
 'baking powder',
 'salt',
 'eggs']

In [7]:
import time
import re
import ast
import os
import requests
import pandas as pd
from tqdm import tqdm

BASE_URL = "http://localhost:11434"
MODEL = "llama3.2:3b"
CHECKPOINT = "main_ing_checkpoint.pkl"
SAVE_EVERY = 25

def ollama_generate(prompt: str, timeout: int = 60, max_retries: int = 5) -> str:
    url = f"{BASE_URL}/api/generate"
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"num_predict": 32, "temperature": 0.0},
    }
    for attempt in range(max_retries):
        try:
            r = requests.post(url, json=payload, timeout=timeout)
            r.raise_for_status()
            return r.json().get("response", "").strip()
        except (requests.exceptions.ReadTimeout,
                requests.exceptions.ConnectTimeout,
                requests.exceptions.ConnectionError,
                requests.exceptions.HTTPError):
            time.sleep(2 ** attempt)
    return ""

_cache = {}

def extract_main_ingredient_one(ingredient: str) -> str:
    s = str(ingredient).strip()
    if not s:
        return ""
    if s in _cache:
        return _cache[s]

    prompt = (
        "Extract the main ingredient name from the string below.\n"
        "Return ONLY the ingredient name, no units, no quantities, no punctuation.\n"
        "Examples:\n"
        "2 cups of flour -> flour\n"
        "1 tsp vanilla extract -> vanilla extract\n"
        "3 eggs -> eggs\n"
        f"String: {s}\n"
        "Answer:"
    )

    ans = ollama_generate(prompt, timeout=60, max_retries=5).strip(" \"'`.\n\t")
    if not ans:
        tmp = re.sub(r"^[\d\s\/\.\(\)\-\–]+", "", s)
        tmp = re.sub(r"\b(cup|cups|tbsp|tsp|teaspoon|tablespoon|oz|g|kg|ml|l|lb|pound|pinch|clove|slices?)\b", "", tmp, flags=re.I)
        tmp = re.sub(r"\s+", " ", tmp).strip(" ,.;:-")
        ans = tmp

    _cache[s] = ans
    return ans

def parse_ingredients_list(cell: str):
    try:
        return ast.literal_eval(cell)
    except Exception:
        return []

In [8]:
if os.path.exists(CHECKPOINT):
    recipes_df = pd.read_pickle(CHECKPOINT)
    if "MainIngredients" not in recipes_df.columns:
        recipes_df["MainIngredients"] = pd.NA
else:
    recipes_df["MainIngredients"] = pd.NA

missing = recipes_df["MainIngredients"].isna()
start_idx = int(missing.idxmax()) if missing.any() else len(recipes_df)
start_idx

450

In [9]:
for i in tqdm(range(start_idx, len(recipes_df)), desc="Rows"):
    items = parse_ingredients_list(recipes_df.at[i, "Ingredients"])
    recipes_df.at[i, "MainIngredients"] = [extract_main_ingredient_one(x) for x in items]
    if (i + 1) % SAVE_EVERY == 0:
        recipes_df.to_pickle(CHECKPOINT)

recipes_df.to_pickle(CHECKPOINT)
recipes_df.head()

Rows: 100%|██████████| 347/347 [49:44<00:00,  8.60s/it] 


,Unnamed: 0,Title,Ingredients,Photo,MainIngredients
0,0,Cookbook:1-2-3-4 Cake,"['1 cup (240 g / 8.5 oz ) butter', '1 cup (240...",https://en.wikibooks.org/wiki/Special:FilePath...,"[butter, milk, vanilla extract, sugar, flour, ..."
1,1,Cookbook:A Nice Cup of Tea,"['32 fl oz (1 L) hot water', '3–5 measures of ...",https://en.wikibooks.org/wiki/Special:FilePath...,"[water, tea, sugar, milk]"
2,2,Cookbook:Aadun (Nigerian Corn Flour with Palm ...,"['1 cup corn flour', '¼ cup palm oil', '½ teas...",https://en.wikibooks.org/wiki/Special:FilePath...,"[corn flour, palm oil, chile pepper, salt]"
3,3,User:Aaron Liu/sandbox,"['5 eggs', '80 grams (⅓ cup ) unsalted margari...",https://en.wikibooks.org/wiki/Special:FilePath...,"[eggs, margarine, salt, sugar, flour, water, y..."
4,4,Cookbook:Abula (Nigerian Three Stews),"['1 cup dried beans', '2 large pieces of meat ...",https://en.wikibooks.org/wiki/Special:FilePath...,"[beans, meat, palm oil, onion, ground crayfish..."


In [10]:
recipes_df.to_csv("recipes_with_main_ingredients.csv", index=False)

In [11]:
recipes_df["Title"]

0                                  Cookbook:1-2-3-4 Cake
1                             Cookbook:A Nice Cup of Tea
2      Cookbook:Aadun (Nigerian Corn Flour with Palm ...
3                                 User:Aaron Liu/sandbox
4                  Cookbook:Abula (Nigerian Three Stews)
                             ...                        
792                                Cookbook:Zucchini Pie
793        Cookbook:Zupa Ogórkowa (Polish Cucumber Soup)
794                               Cookbook:Zuppa Toscana
795    Cookbook:Zwiebelrostbraten with Red Wine Onion...
796      Cookbook:Æbleskiver (Danish Spherical Pancakes)
Name: Title, Length: 797, dtype: object

## Title normalization

In [12]:
import time
import re
import os
import requests
import pandas as pd
from tqdm import tqdm

BASE_URL = "http://localhost:11434"
MODEL = "llama3.2:3b"
CHECKPOINT_TITLE = "norm_title_checkpoint.pkl"
SAVE_EVERY = 50

def ollama_generate(prompt: str, timeout: int = 60, max_retries: int = 5) -> str:
    url = f"{BASE_URL}/api/generate"
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"num_predict": 24, "temperature": 0.0},
    }
    for attempt in range(max_retries):
        try:
            r = requests.post(url, json=payload, timeout=timeout)
            r.raise_for_status()
            return r.json().get("response", "").strip()
        except (requests.exceptions.ReadTimeout,
                requests.exceptions.ConnectTimeout,
                requests.exceptions.ConnectionError,
                requests.exceptions.HTTPError):
            time.sleep(2 ** attempt)
    return ""

def strip_prefix(title: str) -> str:
    s = str(title).strip()
    s = re.sub(r"^(Cookbook|User)\s*:\s*", "", s, flags=re.I)
    s = re.sub(r"\s+", " ", s).strip()
    return s

_title_cache = {}

def normalize_title_one(title: str) -> str:
    s0 = strip_prefix(title)
    if not s0:
        return ""
    if s0 in _title_cache:
        return _title_cache[s0]

    prompt = (
        "Normalize the recipe title.\n"
        "Return ONLY the generalized dish name.\n"
        "Remove origin/nationality adjectives and parenthetical translations.\n"
        "Remove author/user names, file paths, and extra descriptors.\n"
        "If you cannot confidently generalize, return the cleaned original title.\n"
        "Examples:\n"
        "Zupa Ogórkowa (Polish Cucumber Soup) -> Cucumber soup\n"
        "Aadun (Nigerian Corn Flour with Palm Oil) -> Corn flour with palm oil\n"
        "Æbleskiver (Danish Spherical Pancakes) -> Spherical pancakes\n"
        f"Title: {s0}\n"
        "Answer:"
    )

    ans = ollama_generate(prompt, timeout=60, max_retries=5).strip(" \"'`.\n\t")
    if not ans:
        ans = re.sub(r"\s*\([^)]*\)\s*", " ", s0).strip()
        ans = re.sub(r"\s+", " ", ans).strip()

    _title_cache[s0] = ans
    return ans

In [13]:
if os.path.exists(CHECKPOINT_TITLE):
    recipes_df = pd.read_pickle(CHECKPOINT_TITLE)
    if "NormTitle" not in recipes_df.columns:
        recipes_df["NormTitle"] = pd.NA
else:
    recipes_df["NormTitle"] = pd.NA

missing = recipes_df["NormTitle"].isna()
start_idx = int(missing.idxmax()) if missing.any() else len(recipes_df)
start_idx

0

In [14]:
for i in tqdm(range(start_idx, len(recipes_df)), desc="Titles"):
    recipes_df.at[i, "NormTitle"] = normalize_title_one(recipes_df.at[i, "Title"])
    if (i + 1) % SAVE_EVERY == 0:
        recipes_df.to_pickle(CHECKPOINT_TITLE)

recipes_df.to_pickle(CHECKPOINT_TITLE)
recipes_df[["Title", "NormTitle"]].head(20)

Titles: 100%|██████████| 797/797 [16:46<00:00,  1.26s/it]


,Title,NormTitle
0,Cookbook:1-2-3-4 Cake,Cake
1,Cookbook:A Nice Cup of Tea,Nice cup of tea
2,Cookbook:Aadun (Nigerian Corn Flour with Palm ...,Corn flour with palm oil
3,User:Aaron Liu/sandbox,Corn flour with palm oil
4,Cookbook:Abula (Nigerian Three Stews),Three stews
5,Cookbook:Acorn Crusted Salmon,Acorn crusted salmon
6,Cookbook:Adobo Chicken (Philippine),Adobo chicken
7,Cookbook:Affogato,Affogato
8,Cookbook:Afghan Bread,Afghan bread
9,Cookbook:Agedashi Tofu,Agedashi tofu


## Dummy variables transformation

### Main ingredients normalization

In [15]:
import spacy
import re
import os
import pandas as pd
from tqdm import tqdm

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import spacy.cli
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

CHECKPOINT_NORM_ING = "norm_maining_spacy_checkpoint.pkl"
SAVE_EVERY = 25

def clean_phrase(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def lemma_phrase_spacy(phrase: str) -> str:
    p = clean_phrase(phrase)
    if not p:
        return ""
    doc = nlp(p)
    toks = []
    for t in doc:
        if t.is_space or t.is_punct:
            continue
        if t.like_num:
            continue
        if t.pos_ in {"DET"}:
            continue
        lem = t.lemma_.lower().strip()
        if lem in {"-pron-", ""}:
            lem = t.text.lower()
        toks.append(lem)
    out = " ".join(toks).strip()
    return out

canonical = {}

def normalize_list_with_canonical(ingredients_list):
    out = []
    for ing in ingredients_list:
        raw = clean_phrase(ing)
        if not raw:
            out.append("")
            continue
        lem = lemma_phrase_spacy(raw)
        if not lem:
            lem = raw
        if lem in canonical:
            out.append(canonical[lem])
        else:
            canonical[lem] = lem
            out.append(lem)
    return out

In [16]:
if os.path.exists(CHECKPOINT_NORM_ING):
    recipes_df = pd.read_pickle(CHECKPOINT_NORM_ING)
    if "NormalizedMainIngredients" not in recipes_df.columns:
        recipes_df["NormalizedMainIngredients"] = pd.NA
else:
    recipes_df["NormalizedMainIngredients"] = pd.NA

missing = recipes_df["NormalizedMainIngredients"].isna()
start_idx = int(missing.idxmax()) if missing.any() else len(recipes_df)
start_idx

0

In [17]:
for i in tqdm(range(start_idx, len(recipes_df)), desc="NormalizeMainIngredients"):
    recipes_df.at[i, "NormalizedMainIngredients"] = normalize_list_with_canonical(
        recipes_df.at[i, "MainIngredients"]
    )
    if (i + 1) % SAVE_EVERY == 0:
        recipes_df.to_pickle(CHECKPOINT_NORM_ING)

recipes_df.to_pickle(CHECKPOINT_NORM_ING)
recipes_df[["MainIngredients", "NormalizedMainIngredients"]].head(10)

NormalizeMainIngredients: 100%|██████████| 797/797 [00:23<00:00, 34.31it/s]


,MainIngredients,NormalizedMainIngredients
0,"[butter, milk, vanilla extract, sugar, flour, ...","[butter, milk, vanilla extract, sugar, flour, ..."
1,"[water, tea, sugar, milk]","[water, tea, sugar, milk]"
2,"[corn flour, palm oil, chile pepper, salt]","[corn flour, palm oil, chile pepper, salt]"
3,"[eggs, margarine, salt, sugar, flour, water, y...","[egg, margarine, salt, sugar, flour, water, ye..."
4,"[beans, meat, palm oil, onion, ground crayfish...","[bean, meat, palm oil, onion, ground crayfish,..."
5,"[salmon, acorns, egg whites]","[salmon, acorn, egg white]"
6,"[chicken, coconut juice, soy sauce, garlic, ba...","[chicken, coconut juice, soy sauce, garlic, ba..."
7,"[vanilla, espresso]","[vanilla, espresso]"
8,"[dough, Water, yeast, Sugar, flour, Salt, baki...","[dough, water, yeast, sugar, flour, salt, bake..."
9,"[tofu, starch, oil, dashi, soy sauce, mirin, o...","[tofu, starch, oil, dashi, soy sauce, mirin, o..."


In [18]:
recipes_df.columns

Index(['Unnamed: 0', 'Title', 'Ingredients', 'Photo', 'MainIngredients',
       'NormTitle', 'NormalizedMainIngredients'],
      dtype='object')

### Dummy variables

In [19]:
df2 = recipes_df[["Photo", "NormTitle", "NormalizedMainIngredients"]].copy()
df2.head()

,Photo,NormTitle,NormalizedMainIngredients
0,https://en.wikibooks.org/wiki/Special:FilePath...,Cake,"[butter, milk, vanilla extract, sugar, flour, ..."
1,https://en.wikibooks.org/wiki/Special:FilePath...,Nice cup of tea,"[water, tea, sugar, milk]"
2,https://en.wikibooks.org/wiki/Special:FilePath...,Corn flour with palm oil,"[corn flour, palm oil, chile pepper, salt]"
3,https://en.wikibooks.org/wiki/Special:FilePath...,Corn flour with palm oil,"[egg, margarine, salt, sugar, flour, water, ye..."
4,https://en.wikibooks.org/wiki/Special:FilePath...,Three stews,"[bean, meat, palm oil, onion, ground crayfish,..."


In [20]:
df2["NormalizedMainIngredients"] = df2["NormalizedMainIngredients"].apply(
    lambda x: x if isinstance(x, list) else []
)

In [21]:
dummies = (
    df2["NormalizedMainIngredients"]
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
)

dummies = dummies[dummies.ne("")]

In [22]:
X_ing = pd.crosstab(dummies.index, dummies)
X_ing.head()

NormalizedMainIngredients,achiote,acorn,active dry yeast,adobo season,adzuki bean,aekjeot,agushi seed,ajilimo,ajinomoto seasoning,akamu,...,yerba mate,yoghurt,yogurt,yolk,yorkshire pudding,yukon gold potato,zheera,ziti,zucchini,zuckerhut
row_0,,,,,,,,,,,,,,,,,,,,,
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [23]:
df_final = df2.drop(columns=["NormalizedMainIngredients"]).join(X_ing).fillna(0).astype({c: "int8" for c in X_ing.columns})
df_final.head()

,Photo,NormTitle,achiote,acorn,active dry yeast,adobo season,adzuki bean,aekjeot,agushi seed,ajilimo,...,yerba mate,yoghurt,yogurt,yolk,yorkshire pudding,yukon gold potato,zheera,ziti,zucchini,zuckerhut
0,https://en.wikibooks.org/wiki/Special:FilePath...,Cake,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,https://en.wikibooks.org/wiki/Special:FilePath...,Nice cup of tea,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,https://en.wikibooks.org/wiki/Special:FilePath...,Corn flour with palm oil,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,https://en.wikibooks.org/wiki/Special:FilePath...,Corn flour with palm oil,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,https://en.wikibooks.org/wiki/Special:FilePath...,Three stews,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
df_final.to_csv("final_recipes_dataset.csv", index=False)